# Paclitaxel Multi-omics Classification

This notebook evaluates fold-safe supervised classifiers for paclitaxel response prediction using the cleaned RNA, CNV, and mutation matrices from `1-data_preprocessing.ipynb`.

Primary endpoint: a median split of the batch-level growth-curve `angle`, computed once from the full paclitaxel cohort.

- **Resistant**: `angle` below the cohort median
- **Sensitive**: `angle` at or above the cohort median

mRECIST is kept in the metadata as a reference/QC column only -- it is not used to define the modeling endpoint. A median split of `angle` disagrees with the mRECIST-collapsed label (PD/SD=Resistant, PR/CR=Sensitive) on ~24% of models, almost entirely at the ambiguous SD boundary, and RNA expression predicts the angle-based label substantially better than the mRECIST-based one (see Response QC below).

Feature selection is performed inside each cross-validation fold. External gene priors, including the curated paclitaxel gene list and the NEST-VNN gene list, are fixed a priori and force-retained when present.

## Imports and Configuration

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

from pipeline_utils import make_case4_paths

paths = make_case4_paths()
DATA_DIR = paths["proc"]
RESULTS_DIR = paths["results"]

sys.path.append(str(paths["project"] / "4_paclitaxel"))
from feature_sets import (
    add_binary_response,
    load_pax_genes,
    load_nest_genes,
    split_x_class_y,
    select_features_by_omics,
)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
POS_LABEL = 1  # Sensitive

print("Project root:", paths["project"])
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Project root: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study
Data dir: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/data/procdata/4_paclitaxel
Results dir: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/results/4_paclitaxel


## Load Data and Define Binary Response

In [2]:
metadata_df = pd.read_csv(DATA_DIR / "metadata.csv", index_col=0)

# Fixed threshold from the FULL 80-model paclitaxel cohort, applied identically
# to every omics subset below regardless of its sample count -- rna_all (67),
# rna_paired/cnv/mutation (62 each) would each shift the cutoff if the median
# were recomputed per subset instead of reusing this one shared value.
ANGLE_MEDIAN = metadata_df["angle"].median()
print(f"Angle median (full {len(metadata_df)}-model cohort): {ANGLE_MEDIAN:.2f}")

metadata_df = add_binary_response(metadata_df, angle_median=ANGLE_MEDIAN)
rna_df = add_binary_response(pd.read_csv(DATA_DIR / "rna_logtpm.csv", index_col=0), angle_median=ANGLE_MEDIAN)
cnv_df = add_binary_response(pd.read_csv(DATA_DIR / "cnv.csv", index_col=0), angle_median=ANGLE_MEDIAN)
mutation_df = add_binary_response(pd.read_csv(DATA_DIR / "mutation.csv", index_col=0), angle_median=ANGLE_MEDIAN)

pax_genes = load_pax_genes()
nest_genes = load_nest_genes()

all3_ids = sorted(set(rna_df.index) & set(cnv_df.index) & set(mutation_df.index))
rna_paired_df = rna_df.loc[all3_ids]
cnv_paired_df = cnv_df.loc[all3_ids]
mutation_paired_df = mutation_df.loc[all3_ids]

for name, df in [
    ("metadata", metadata_df),
    ("rna_all", rna_df),
    ("rna_paired", rna_paired_df),
    ("cnv", cnv_paired_df),
    ("mutation", mutation_paired_df),
]:
    print(f"\n{name} {df.shape}")
    print(df["mRECIST"].value_counts().to_string())
    print(df["response_binary"].value_counts().to_string())

print(f"\nPaclitaxel prior genes: {len(pax_genes)}")
print("NEST prior genes:", len(nest_genes))


Angle median (full 80-model cohort): 68.23



metadata (80, 4)
mRECIST
SD    29
CR    22
PD    18
PR    11
response_binary
Resistant    40
Sensitive    40

rna_all (67, 49352)
mRECIST
SD    23
CR    20
PD    15
PR     9
response_binary
Sensitive    34
Resistant    33

rna_paired (62, 49352)
mRECIST
SD    21
CR    19
PD    14
PR     8
response_binary
Sensitive    32
Resistant    30

cnv (62, 10317)
mRECIST
SD    21
CR    19
PD    14
PR     8
response_binary
Sensitive    32
Resistant    30

mutation (62, 13033)
mRECIST
SD    21
CR    19
PD    14
PR     8
response_binary
Sensitive    32
Resistant    30

Paclitaxel prior genes: 69
NEST prior genes: 718


## Response QC

How does the angle-median-split `response_binary` (the actual modeling endpoint) relate to the original mRECIST clinical categories, kept in the metadata as a reference column only?

In [3]:
MRECIST_ORDER_QC = ["PD", "SD", "PR", "CR"]

concordance = pd.crosstab(metadata_df["mRECIST"], metadata_df["response_binary"]).reindex(MRECIST_ORDER_QC)
print("mRECIST category x angle-median-split response_binary:")
print(concordance)

# mRECIST-collapsed reference: PD/SD -> Resistant, PR/CR -> Sensitive. This is
# not a ground truth -- angle was chosen over mRECIST deliberately (see intro
# markdown) -- it just documents where the two definitions agree/disagree.
mrecist_collapsed_map = {"PD": "Resistant", "SD": "Resistant", "PR": "Sensitive", "CR": "Sensitive"}
mrecist_collapsed = metadata_df["mRECIST"].map(mrecist_collapsed_map)
agreement = (metadata_df["response_binary"] == mrecist_collapsed).mean()
print(f"\nAgreement with mRECIST-collapsed (PD/SD=Resistant, PR/CR=Sensitive): {agreement:.1%}")

flipped = metadata_df.loc[
    metadata_df["response_binary"] != mrecist_collapsed, ["mRECIST", "angle", "response_binary"]
].sort_values("angle")
print(f"\n{len(flipped)} models where the angle-median-split label disagrees with mRECIST-collapsed:")

response_qc_df = flipped.reset_index()
response_qc_df.to_csv(RESULTS_DIR / "response_metric_qc.csv", index=False)
response_qc_df


mRECIST category x angle-median-split response_binary:
response_binary  Resistant  Sensitive
mRECIST                              
PD                      16          2
SD                      18         11
PR                       4          7
CR                       2         20

Agreement with mRECIST-collapsed (PD/SD=Resistant, PR/CR=Sensitive): 76.2%

19 models where the angle-median-split label disagrees with mRECIST-collapsed:


,model.id,mRECIST,angle,response_binary
0,NOTCH10,PR,24.952204,Resistant
1,REF024,PR,35.034703,Resistant
2,S18674,CR,43.546818,Resistant
3,S73230,CR,46.725173,Resistant
4,S44721,PR,49.658143,Resistant
5,S241417,PR,66.756904,Resistant
6,S44205,SD,69.696698,Sensitive
7,REF034,SD,70.657522,Sensitive
8,S71335,SD,88.056448,Sensitive
9,DCPTOB19,SD,106.370493,Sensitive


## Classifier and Metric Helpers

In [4]:
def classification_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "auroc": roc_auc_score(y_true, y_score),
        "auprc": average_precision_score(y_true, y_score),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def make_classification_models(random_state=RANDOM_STATE):
    return {
        "logistic_l2": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l2",
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                )),
            ]),
            {"model__C": [0.01, 0.1, 1.0, 10.0]},
        ),
        "logistic_l1": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    penalty="l1",
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=random_state,
                )),
            ]),
            {"model__C": [0.01, 0.1, 1.0, 10.0]},
        ),
        "svm_rbf": (
            Pipeline([
                ("scale", StandardScaler()),
                ("model", SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight="balanced",
                    random_state=random_state,
                )),
            ]),
            {
                "model__C": [0.1, 1.0, 10.0],
                "model__gamma": ["scale", 0.01, 0.001],
            },
        ),
        "random_forest": (
            RandomForestClassifier(
                n_estimators=500,
                max_features="sqrt",
                class_weight="balanced",
                random_state=random_state,
                n_jobs=-1,
            ),
            {
                "max_depth": [None, 3, 5],
                "min_samples_leaf": [2, 3, 5],
            },
        ),
    }


def tune_model(model, param_grid, X_train, y_train):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_params_


## Single-omics Evaluation

In [5]:
def evaluate_single_omics(name, df, prior_mode="pax"):
    X, y = split_x_class_y(df)
    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    models = make_classification_models()

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        selected = select_features_by_omics(name, X_train_raw, prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        X_train = X_train_raw[selected]
        X_test = X_test_raw[selected]

        for model_name, (model, grid) in models.items():
            best_model, best_params = tune_model(clone(model), grid, X_train, y_train)
            y_pred = best_model.predict(X_test)
            y_score = positive_class_score(best_model, X_test)
            metrics = classification_metrics(y_test, y_pred, y_score)

            rows.append({
                "analysis": "single_omics",
                "omics": name,
                "prior_mode": prior_mode,
                "model": model_name,
                "fold": fold_id,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_features": len(selected),
                "n_pax_genes": len(set(selected) & pax_genes),
                "n_nest_genes": len(set(selected) & nest_genes),
                "best_params": str(best_params),
                **metrics,
            })

            predictions.extend([
                {
                    "analysis": "single_omics",
                    "omics": name,
                    "prior_mode": prior_mode,
                    "model": model_name,
                    "fold": fold_id,
                    "sample_id": sample_id,
                    "y_true": int(true),
                    "y_true_label": "Sensitive" if true == 1 else "Resistant",
                    "y_pred": int(pred),
                    "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                    "y_score_sensitive": score,
                }
                for sample_id, true, pred, score in zip(X_test.index, y_test, y_pred, y_score)
            ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


## Run Single-omics Baselines

This evaluates RNA on all available RNA samples, and RNA/CNV/mutation on the 62-sample paired cohort for fair comparison.

In [6]:
single_results = []
single_predictions = []

single_jobs = [
    ("rna_all", rna_df),
    ("rna_paired", rna_paired_df),
    ("cnv", cnv_paired_df),
    ("mutation", mutation_paired_df),
]

for name, df in single_jobs:
    print(f"Running {name}...")
    result_df, pred_df = evaluate_single_omics(name, df, prior_mode="pax")
    single_results.append(result_df)
    single_predictions.append(pred_df)

single_results_df = pd.concat(single_results, ignore_index=True)
single_predictions_df = pd.concat(single_predictions, ignore_index=True)

single_results_df.to_csv(RESULTS_DIR / "single_omics_classification_cv_results.csv", index=False)
single_predictions_df.to_csv(RESULTS_DIR / "single_omics_classification_cv_predictions.csv", index=False)

single_results_df.groupby(["omics", "model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running rna_all...


Running rna_paired...


Running cnv...


Running mutation...


auroc  auprc  balanced_accuracy     f1  sensitivity  \
omics      model                                                                
cnv        logistic_l1    0.507  0.603              0.552  0.447        0.431   
           logistic_l2    0.529  0.616              0.521  0.518        0.535   
           random_forest  0.559  0.652              0.550  0.571        0.607   
           svm_rbf        0.407  0.541              0.527  0.428        0.534   
mutation   logistic_l1    0.460  0.508              0.476  0.058        0.059   
           logistic_l2    0.358  0.487              0.376  0.331        0.346   
           random_forest  0.396  0.493              0.448  0.454        0.495   
           svm_rbf        0.582  0.631              0.497  0.451        0.640   
rna_all    logistic_l1    0.575  0.627              0.559  0.541        0.535   
           logistic_l2    0.673  0.710              0.631  0.617        0.611   
           random_forest  0.732  0.757              0.670  0.668        0.686   
           svm_rbf        0.532  0.610              0.570  0.516        0.564   
rna_paired logistic_l1    0.564  0.627              0.545  0.510        0.524   
           logistic_l2    0.683  0.722              0.650  0.667        0.708   
           random_forest  0.700  0.742              0.666  0.679        0.718   
           svm_rbf        0.467  0.576              0.580  0.505        0.552   

                          specificity  n_features  
omics      model                                   
cnv        logistic_l1          0.673      491.60  
           logistic_l2          0.507      491.60  
           random_forest        0.493      491.60  
           svm_rbf              0.520      491.60  
mutation   logistic_l1          0.893      426.60  
           logistic_l2          0.407      426.60  
           random_forest        0.400      426.60  
           svm_rbf              0.353      426.60  
rna_all    logistic_l1          0.582     2054.88  
           logistic_l2          0.651     2054.88  
           random_forest        0.655     2054.88  
           svm_rbf              0.577     2054.88  
rna_paired logistic_l1          0.567     2055.52  
           logistic_l2          0.593     2055.52  
           random_forest        0.613     2055.52  
           svm_rbf              0.607     2055.52

## Early-fusion Multi-omics Evaluation

For each fold, feature selection is performed independently within RNA, CNV, and mutation training folds. The selected matrices are concatenated and evaluated as a single feature table.

In [7]:
def evaluate_early_fusion(prior_mode="pax"):
    X_rna, y = split_x_class_y(rna_paired_df)
    X_cnv, _ = split_x_class_y(cnv_paired_df)
    X_mut, _ = split_x_class_y(mutation_paired_df)

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )
    models = make_classification_models()

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_rna, y), start=1):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        selected_rna = select_features_by_omics("rna_paired", X_rna.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        selected_cnv = select_features_by_omics("cnv", X_cnv.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)
        selected_mut = select_features_by_omics("mutation", X_mut.iloc[train_idx], prior_mode=prior_mode, pax_genes=pax_genes, nest_genes=nest_genes)

        X_train = pd.concat(
            [
                X_rna.iloc[train_idx][selected_rna].add_prefix("rna__"),
                X_cnv.iloc[train_idx][selected_cnv].add_prefix("cnv__"),
                X_mut.iloc[train_idx][selected_mut].add_prefix("mut__"),
            ],
            axis=1,
        )
        X_test = pd.concat(
            [
                X_rna.iloc[test_idx][selected_rna].add_prefix("rna__"),
                X_cnv.iloc[test_idx][selected_cnv].add_prefix("cnv__"),
                X_mut.iloc[test_idx][selected_mut].add_prefix("mut__"),
            ],
            axis=1,
        )

        for model_name, (model, grid) in models.items():
            best_model, best_params = tune_model(clone(model), grid, X_train, y_train)
            y_pred = best_model.predict(X_test)
            y_score = positive_class_score(best_model, X_test)
            metrics = classification_metrics(y_test, y_pred, y_score)

            rows.append({
                "analysis": "early_fusion",
                "omics": "rna_cnv_mutation",
                "prior_mode": prior_mode,
                "model": model_name,
                "fold": fold_id,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_features": X_train.shape[1],
                "n_rna_features": len(selected_rna),
                "n_cnv_features": len(selected_cnv),
                "n_mutation_features": len(selected_mut),
                "best_params": str(best_params),
                **metrics,
            })

            predictions.extend([
                {
                    "analysis": "early_fusion",
                    "omics": "rna_cnv_mutation",
                    "prior_mode": prior_mode,
                    "model": model_name,
                    "fold": fold_id,
                    "sample_id": sample_id,
                    "y_true": int(true),
                    "y_true_label": "Sensitive" if true == 1 else "Resistant",
                    "y_pred": int(pred),
                    "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                    "y_score_sensitive": score,
                }
                for sample_id, true, pred, score in zip(X_test.index, y_test, y_pred, y_score)
            ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [8]:
print("Running early fusion...")
early_results_df, early_predictions_df = evaluate_early_fusion(prior_mode="pax")

early_results_df.to_csv(RESULTS_DIR / "early_fusion_classification_cv_results.csv", index=False)
early_predictions_df.to_csv(RESULTS_DIR / "early_fusion_classification_cv_predictions.csv", index=False)

early_results_df.groupby(["model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running early fusion...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
logistic_l1,0.562,0.619,0.537,0.554,0.581,0.493,2973.72
logistic_l2,0.631,0.694,0.577,0.595,0.628,0.527,2973.72
random_forest,0.683,0.723,0.656,0.671,0.719,0.593,2973.72
svm_rbf,0.419,0.540,0.524,0.436,0.548,0.500,2973.72


## Late-fusion / Stacking Multi-omics Evaluation

Late fusion trains one base classifier per omics layer and combines their out-of-fold probability predictions with a simple meta-classifier. This tests whether CNV and mutation add complementary information without concatenating thousands of features directly with RNA.

For runtime, the default stack uses `logistic_l2` base models and a ridge-like logistic meta-model. The implementation keeps feature selection inside the relevant training folds.

In [9]:
def get_model_spec(model_name):
    models = make_classification_models()
    if model_name not in models:
        raise ValueError(f"Unknown model_name: {model_name}")
    return models[model_name]


def fit_predict_one_omics(omics_name, X_train_raw, y_train, X_test_raw, model_name, prior_mode):
    selected = select_features_by_omics(
        omics_name,
        X_train_raw,
        prior_mode=prior_mode,
        pax_genes=pax_genes,
        nest_genes=nest_genes,
    )
    model, grid = get_model_spec(model_name)
    best_model, best_params = tune_model(clone(model), grid, X_train_raw[selected], y_train)
    test_score = positive_class_score(best_model, X_test_raw[selected])
    return test_score, selected, best_params


def build_oof_base_predictions(X_layers, y_train, base_model_name="logistic_l2", prior_mode="pax"):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    oof = pd.DataFrame(index=y_train.index)

    for omics_name, X_train_raw in X_layers.items():
        scores = pd.Series(index=y_train.index, dtype=float)
        for inner_train_idx, inner_valid_idx in inner_cv.split(X_train_raw, y_train):
            X_inner_train = X_train_raw.iloc[inner_train_idx]
            X_inner_valid = X_train_raw.iloc[inner_valid_idx]
            y_inner_train = y_train.iloc[inner_train_idx]

            selected = select_features_by_omics(
                omics_name,
                X_inner_train,
                prior_mode=prior_mode,
                pax_genes=pax_genes,
                nest_genes=nest_genes,
            )
            model, grid = get_model_spec(base_model_name)
            best_model, _ = tune_model(
                clone(model),
                grid,
                X_inner_train[selected],
                y_inner_train,
            )
            scores.iloc[inner_valid_idx] = positive_class_score(best_model, X_inner_valid[selected])

        oof[omics_name] = scores

    return oof


def evaluate_late_fusion(base_model_name="logistic_l2", prior_mode="pax"):
    X_rna, y = split_x_class_y(rna_paired_df)
    X_cnv, _ = split_x_class_y(cnv_paired_df)
    X_mut, _ = split_x_class_y(mutation_paired_df)

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )

    meta_model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            C=1.0,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])

    rows = []
    predictions = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X_rna, y), start=1):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_layers = {
            "rna_paired": X_rna.iloc[train_idx],
            "cnv": X_cnv.iloc[train_idx],
            "mutation": X_mut.iloc[train_idx],
        }
        test_layers = {
            "rna_paired": X_rna.iloc[test_idx],
            "cnv": X_cnv.iloc[test_idx],
            "mutation": X_mut.iloc[test_idx],
        }

        meta_train = build_oof_base_predictions(
            train_layers,
            y_train,
            base_model_name=base_model_name,
            prior_mode=prior_mode,
        )

        meta_test = pd.DataFrame(index=y_test.index)
        feature_counts = {}
        pax_counts = {}
        nest_counts = {}
        base_params = {}

        for omics_name in ["rna_paired", "cnv", "mutation"]:
            test_score, selected, best_params = fit_predict_one_omics(
                omics_name,
                train_layers[omics_name],
                y_train,
                test_layers[omics_name],
                model_name=base_model_name,
                prior_mode=prior_mode,
            )
            meta_test[omics_name] = test_score
            feature_counts[omics_name] = len(selected)
            pax_counts[omics_name] = len(set(selected) & pax_genes)
            nest_counts[omics_name] = len(set(selected) & nest_genes)
            base_params[omics_name] = best_params

        meta_model.fit(meta_train, y_train)
        y_score = positive_class_score(meta_model, meta_test)
        y_pred = meta_model.predict(meta_test)
        metrics = classification_metrics(y_test, y_pred, y_score)

        rows.append({
            "analysis": "late_fusion_stacking",
            "omics": "rna_cnv_mutation",
            "prior_mode": prior_mode,
            "model": f"stack_{base_model_name}",
            "fold": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_features": meta_train.shape[1],
            "n_rna_features": feature_counts["rna_paired"],
            "n_cnv_features": feature_counts["cnv"],
            "n_mutation_features": feature_counts["mutation"],
            "n_rna_pax_genes": pax_counts["rna_paired"],
            "n_cnv_pax_genes": pax_counts["cnv"],
            "n_mutation_pax_genes": pax_counts["mutation"],
            "base_model": base_model_name,
            "base_params": str(base_params),
            **metrics,
        })

        predictions.extend([
            {
                "analysis": "late_fusion_stacking",
                "omics": "rna_cnv_mutation",
                "prior_mode": prior_mode,
                "model": f"stack_{base_model_name}",
                "fold": fold_id,
                "sample_id": sample_id,
                "y_true": int(true),
                "y_true_label": "Sensitive" if true == 1 else "Resistant",
                "y_pred": int(pred),
                "y_pred_label": "Sensitive" if pred == 1 else "Resistant",
                "y_score_sensitive": score,
                "rna_score": meta_test.loc[sample_id, "rna_paired"],
                "cnv_score": meta_test.loc[sample_id, "cnv"],
                "mutation_score": meta_test.loc[sample_id, "mutation"],
            }
            for sample_id, true, pred, score in zip(y_test.index, y_test, y_pred, y_score)
        ])

    return pd.DataFrame(rows), pd.DataFrame(predictions)


In [10]:
print("Running late fusion / stacking...")
late_results_df, late_predictions_df = evaluate_late_fusion(
    base_model_name="logistic_l2",
    prior_mode="pax",
)

late_results_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_results.csv", index=False)
late_predictions_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_predictions.csv", index=False)

late_results_df.groupby(["model"])[[
    "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
]].mean().round(3)


Running late fusion / stacking...


,auroc,auprc,balanced_accuracy,f1,sensitivity,specificity,n_features
model,,,,,,,
stack_logistic_l2,0.626,0.684,0.596,0.588,0.606,0.587,3.0


In [11]:
RUN_STACKING_SENSITIVITY = True

if RUN_STACKING_SENSITIVITY:

    late_results = []
    late_predictions = []

    for base_model_name in ["logistic_l2", "random_forest", "svm_rbf"]:
        print(f"Running late fusion / stacking with {base_model_name}...")
        result_df, pred_df = evaluate_late_fusion(
            base_model_name=base_model_name,
            prior_mode="pax",
        )
        late_results.append(result_df)
        late_predictions.append(pred_df)

    late_results_df = pd.concat(late_results, ignore_index=True)
    late_predictions_df = pd.concat(late_predictions, ignore_index=True)

    late_results_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_results.csv", index=False)
    late_predictions_df.to_csv(RESULTS_DIR / "late_fusion_classification_cv_predictions.csv", index=False)

    late_results_df.groupby(["model"])[[
        "auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "n_features"
    ]].mean().round(3)

Running late fusion / stacking with logistic_l2...


Running late fusion / stacking with random_forest...


Running late fusion / stacking with svm_rbf...


## Compare Model Families

In [12]:
from scipy.stats import ttest_1samp
from scipy.stats import t as t_dist


def fdr_bh(pvals):
    """Benjamini-Hochberg FDR correction."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    return out


all_results_df = pd.concat([single_results_df, early_results_df, late_results_df], ignore_index=True)

# Per model/omics combination: mean AUROC across repeated-CV folds, a 95% t-based
# CI on that mean, and a one-sample t-test of "mean fold AUROC != 0.5 (chance)".
# Caveat: repeated-CV folds are not independent (they reuse the same ~62-80
# samples across repeats), so this t-test is the standard but slightly
# anti-conservative simplification used across most ML papers reporting
# repeated-CV significance, not a variance-corrected test (e.g. Nadeau-Bengio).
# Treat p-values here as indicative, and always read them alongside the CI.
summary_rows = []
for (analysis, omics, model), group in all_results_df.groupby(["analysis", "omics", "model"]):
    aurocs = group["auroc"].to_numpy()
    n_folds = len(aurocs)
    mean = aurocs.mean()
    sd = aurocs.std(ddof=1)
    se = sd / np.sqrt(n_folds)
    tcrit = t_dist.ppf(0.975, df=n_folds - 1)
    t_stat, p_val = ttest_1samp(aurocs, 0.5)

    summary_rows.append({
        "analysis": analysis,
        "omics": omics,
        "model": model,
        "auroc_mean": mean,
        "auroc_sd": sd,
        "auroc_ci_lo": mean - tcrit * se,
        "auroc_ci_hi": mean + tcrit * se,
        "auroc_p_vs_chance": p_val,
        "auprc_mean": group["auprc"].mean(),
        "auprc_sd": group["auprc"].std(),
        "balanced_accuracy_mean": group["balanced_accuracy"].mean(),
        "balanced_accuracy_sd": group["balanced_accuracy"].std(),
        "f1_mean": group["f1"].mean(),
        "sensitivity_mean": group["sensitivity"].mean(),
        "specificity_mean": group["specificity"].mean(),
        "n_features_mean": group["n_features"].mean(),
        "n_folds": n_folds,
    })

summary_df = pd.DataFrame(summary_rows)

# Benjamini-Hochberg FDR correction across all model/omics combinations tested here
summary_df["auroc_p_adj_fdr"] = fdr_bh(summary_df["auroc_p_vs_chance"].to_numpy())

summary_df = summary_df.sort_values(["auroc_mean", "auprc_mean"], ascending=False)

n_sig = (summary_df["auroc_p_adj_fdr"] < 0.05).sum()
print(f"{n_sig}/{len(summary_df)} model/omics combinations significantly better than chance after FDR correction (q < 0.05)")

summary_df.to_csv(RESULTS_DIR / "classification_model_summary.csv", index=False)
summary_df.round(3)

15/23 model/omics combinations significantly better than chance after FDR correction (q < 0.05)


,analysis,omics,model,auroc_mean,auroc_sd,auroc_ci_lo,auroc_ci_hi,auroc_p_vs_chance,auprc_mean,auprc_sd,balanced_accuracy_mean,balanced_accuracy_sd,f1_mean,sensitivity_mean,specificity_mean,n_features_mean,n_folds,auroc_p_adj_fdr
17,single_omics,rna_all,random_forest,0.732,0.117,0.684,0.780,0.000,0.757,0.114,0.670,0.113,0.668,0.686,0.655,2054.88,25,0.000
21,single_omics,rna_paired,random_forest,0.700,0.132,0.645,0.754,0.000,0.742,0.104,0.666,0.153,0.679,0.718,0.613,2055.52,25,0.000
2,early_fusion,rna_cnv_mutation,random_forest,0.683,0.126,0.632,0.735,0.000,0.723,0.104,0.656,0.123,0.671,0.719,0.593,2973.72,25,0.000
20,single_omics,rna_paired,logistic_l2,0.683,0.130,0.630,0.737,0.000,0.722,0.110,0.650,0.126,0.667,0.708,0.593,2055.52,25,0.000
16,single_omics,rna_all,logistic_l2,0.673,0.137,0.617,0.729,0.000,0.710,0.115,0.631,0.121,0.617,0.611,0.651,2054.88,25,0.000
5,late_fusion_stacking,rna_cnv_mutation,stack_random_forest,0.671,0.181,0.597,0.746,0.000,0.725,0.151,0.612,0.137,0.605,0.610,0.613,3.00,25,0.000
1,early_fusion,rna_cnv_mutation,logistic_l2,0.631,0.136,0.575,0.687,0.000,0.694,0.097,0.577,0.117,0.595,0.628,0.527,2973.72,25,0.000
4,late_fusion_stacking,rna_cnv_mutation,stack_logistic_l2,0.626,0.151,0.563,0.688,0.000,0.684,0.128,0.596,0.120,0.588,0.606,0.587,3.00,25,0.001
14,single_omics,mutation,svm_rbf,0.582,0.140,0.524,0.640,0.008,0.631,0.117,0.497,0.038,0.451,0.640,0.353,426.60,25,0.016
15,single_omics,rna_all,logistic_l1,0.575,0.145,0.515,0.634,0.016,0.627,0.117,0.559,0.126,0.541,0.535,0.582,2054.88,25,0.025


## Notes for Interpretation

Recommended first-pass readout:

1. Compare `rna_paired` vs `cnv` vs `mutation` to establish single-omics baselines.
2. Compare `early_fusion` against the best single-omics model.
3. If early fusion does not improve, use this as support for late-fusion/stacking as the next model family.
4. Use the saved fold-level predictions for paired statistical comparisons and ROC/PR plots.